# Dynabatch sampler vs Seq2Seq Comparison Training NLLB200 - 600M 

## Install packages

In [ ]:
# # # !pip install torch==2.11.0
# !pip install datasets
# !pip install transformers==5.5.3
# !pip install pandas
# !pip install tqdm
# !pip install "flash-attn @ https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3+cu130torch2.11-cp312-cp312-linux_x86_64.whl"
# !pip install accelerate
# # !pip install dynabatch

## Imports

In [1]:
from copy import deepcopy
import torch
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainer
from transformers.training_args_seq2seq import Seq2SeqTrainingArguments
from datasets import DatasetDict
from datasets import Dataset

from dynabatch import dynabatch_sampler, MemoryCleanupCallback, make_dynabatch_trainer
import pandas as pd
import gc
from transformers import TrainerCallback
from datetime import datetime

## Load Data

In [2]:
train_samples = 100000
val_samples = 1000

# Picking Brenton language because NLLB 200 wasnt trained on it
dataset = load_dataset("Helsinki-NLP/opus-100", "br-en")

train_data = dataset['train'].shuffle(seed=42).select(range(train_samples)).to_pandas()
val_data = dataset['validation'].shuffle(seed=42).select(range(val_samples)).to_pandas()

train_data[['brenton', 'english']] = train_data['translation'].apply(pd.Series)
val_data[['brenton', 'english']] = val_data['translation'].apply(pd.Series)

train_input_english = train_data['english'].tolist()
train_output_brenton = train_data['brenton'].tolist()

val_input_english = val_data['english'].tolist()
val_output_brenton = val_data['brenton'].tolist()

## Configs

In [3]:
model_id = "facebook/nllb-200-distilled-600M"
batch_size = 10
test_batch_size = 10

max_length = 512

# training arguments
output_dir = "results"
learning_rate = 2e-5
warmup_ratio = 0.2
weight_decay = 0.01
epochs = 3
dataloader_num_workers = 4
eval_steps = 300
logging_steps = 300
accumulation_steps = 8

training_arguments = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=test_batch_size,
    weight_decay=weight_decay,
    num_train_epochs=epochs,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  
    dataloader_pin_memory=True,
    dataloader_num_workers=dataloader_num_workers,
    save_strategy="no",
    remove_unused_columns=True,
    lr_scheduler_type="cosine",
    warmup_ratio=warmup_ratio, 
    gradient_accumulation_steps=accumulation_steps,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=eval_steps,
    logging_strategy="steps",
    logging_steps=logging_steps,
)

if torch.cuda.is_available():
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU found. Change runtime type to GPU.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Using GPU: NVIDIA GeForce RTX 5090


## Load Model

In [4]:
def load_model():
    return AutoModelForSeq2SeqLM.from_pretrained(
            model_id,
            attn_implementation="flash_attention_2",
        )

def load_tokenizer():
    return AutoTokenizer.from_pretrained(model_id, src_lang="eng_Latn", tgt_lang="bre_Latn")

MODEL = load_model()
TOKENIZER = load_tokenizer()

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in M2M100ForConditionalGeneration is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in M2M100Model is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dy

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

## Tokenize Data

In [5]:
def tokenize_function(examples):
    english_texts = examples['english']
    brenton_texts = examples['brenton']

    english_tokens = TOKENIZER(english_texts, max_length=max_length, truncation=True)
    brenton_tokens = TOKENIZER(brenton_texts, max_length=max_length, truncation=True)

    return {
        "input_ids": english_tokens["input_ids"],
        "attention_mask": english_tokens["attention_mask"],
        "labels": brenton_tokens["input_ids"],
    }
    
datasets = DatasetDict({"train": Dataset.from_pandas(train_data), "val": Dataset.from_pandas(val_data)})
datasets = datasets.map(
        tokenize_function,
        num_proc=0,
        remove_columns=datasets["train"].column_names,
        batched=True,
        batch_size=100,
        desc="tokenizing",
    )

tokenizing:   0%|          | 0/100000 [00:00<?, ? examples/s]

tokenizing:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Load Dynabatch train sampler

In [6]:
dynabatch_train_sampler = dynabatch_sampler(
        texts=train_input_english,
        tokenizer=TOKENIZER,
        batch_size=batch_size,
        max_input_token_length=max_length,
        max_batch_range=5.0, # Increase batch size potentially upto 5x (default 2.0)
        shuffle=True,
        shuffle_seed=21,
    )

Step 1: Get Lengths (num_proc=4):   0%|          | 0/100000 [00:00<?, ? examples/s]

Step 2: building dynamic batches: 100%|██████████| 100000/100000 [00:27<00:00, 3636.48seq/s]


## Trainers and other helper functions

In [7]:
def get_seq2seq_data_collator():
    return DataCollatorForSeq2Seq(
        tokenizer=TOKENIZER, 
        model=MODEL, 
        padding=True,
        return_tensors="pt",
        pad_to_multiple_of=8,
    )
    

def get_dynabatch_training_arguments(args):
    """
    This function auto-scales the eval_steps and logging_steps of the training arguments
    because we want to call evaluation for dynabatch trainer the same number of time as Seq2SeqTrainer so the training time comparison is fair
    """
    dynbatch_args = deepcopy(args)
    total_steps_per_epoch = len(datasets["train"]) // batch_size
    total_steps_per_epoch_dynabatch = len(dynabatch_train_sampler)
    multiplier = total_steps_per_epoch_dynabatch / total_steps_per_epoch 
    dynbatch_args.eval_steps = int(dynbatch_args.eval_steps * multiplier)
    dynbatch_args.logging_steps = int(dynbatch_args.logging_steps * multiplier)
    return dynbatch_args


def display_args(args, trainer, learning_rate):
    print("*"*35)
    print(f"N steps per epoch:\t{len(trainer.get_train_dataloader())}")
    print(f"Learning rate:\t\t{learning_rate:6f}")
    print(f"Eval steps:\t\t{args.eval_steps}")
    print(f"Logging steps:\t\t{args.logging_steps}")
    print("*"*35)


def get_dynabatch_trainer(model):
    dynabatch_training_arguments = get_dynabatch_training_arguments(training_arguments)

    DynabatchSeq2SeqTrainer = make_dynabatch_trainer(Seq2SeqTrainer)
    trainer = DynabatchSeq2SeqTrainer(
        dynabatch_sampler=dynabatch_train_sampler,
        model=model,
        args=dynabatch_training_arguments,
        # Since dynabatch will have lesser step per epoch
        # we auto-scales LR to have similar gradient updates as Seq2SeqTrainer to have a fair comparison
        auto_scale_lr=True, 
        train_dataset=datasets["train"],
        eval_dataset=datasets["val"],
        processing_class=TOKENIZER,
        data_collator=get_seq2seq_data_collator(), 
        callbacks=[MemoryCleanupCallback()],
        # if OOM, split the batch into smaller chunks and retry
        oom_fallback="split_retry", 
        oom_min_batch_size=batch_size,
    )
    display_args(dynabatch_training_arguments, trainer, trainer.args.learning_rate)
    return trainer


def get_seq2seq_trainer(model):
    trainer = Seq2SeqTrainer(
            model=model,
            args=training_arguments,
            train_dataset=datasets["train"],
            eval_dataset=datasets["val"],
            processing_class=TOKENIZER,
            data_collator=get_seq2seq_data_collator(),
        )
    display_args(training_arguments, trainer, training_arguments.learning_rate)
    return trainer


## Train using Dyanbatch

In [8]:
dynabatch_trainer = get_dynabatch_trainer(MODEL)
start_time = datetime.now()
dynabatch_trainer.train()
time_taken_dynabatch = datetime.now() - start_time

***********************************
N steps per epoch:	2635
Learning rate:		0.000076
Eval steps:		79
Logging steps:		79
***********************************


Step,Training Loss,Validation Loss
79,223.636619,5.585829
158,133.154544,4.228954
237,108.139735,3.737454
316,97.081092,3.520431
395,88.820869,3.407780
474,86.116891,3.297354
553,83.812982,3.239007
632,83.365000,3.203167
711,79.604319,3.182421
790,78.961778,3.168615


OOM fallback engaged: retrying training step in 4 chunk(s) of <= 10 samples.
OOM fallback engaged: retrying training step in 4 chunk(s) of <= 10 samples.
OOM fallback engaged: retrying training step in 4 chunk(s) of <= 10 samples.
OOM fallback engaged: retrying training step in 4 chunk(s) of <= 10 samples.
OOM fallback engaged: retrying training step in 4 chunk(s) of <= 10 samples.
OOM fallback engaged: retrying training step in 4 chunk(s) of <= 10 samples.
OOM fallback engaged: retrying training step in 4 chunk(s) of <= 10 samples.
OOM fallback engaged: retrying training step in 4 chunk(s) of <= 10 samples.
OOM fallback engaged: retrying training step in 4 chunk(s) of <= 10 samples.


## Train using default Seq2SeqTrainer + DataCollatorForSeq2Seq

In [9]:
# delete Model trained by Dynabatch Trainer
del MODEL, dynabatch_trainer
gc.collect()
torch.cuda.empty_cache()
MODEL = load_model() # reload a fresh untrained model

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [10]:
seq2seq_trainer = get_seq2seq_trainer(MODEL)
start_time = datetime.now()
seq2seq_trainer.train()
time_taken_seq2seq = datetime.now() - start_time

***********************************
N steps per epoch:	10000
Learning rate:		0.000020
Eval steps:		300
Logging steps:		300
***********************************


Step,Training Loss,Validation Loss
300,58.369428,5.582905
600,37.521260,4.430284
900,31.152012,3.950919
1200,28.129469,3.693409
1500,26.390412,3.540449
1800,25.365830,3.440381
2100,24.614440,3.376329
2400,24.172510,3.323525
2700,23.672656,3.297401
3000,23.430563,3.281006


## Comparison

In [11]:
print(f"Time taken for Dynabatch training:\t{(time_taken_dynabatch).total_seconds()}")
print(f"Time taken for Seq2Seq training:\t{(time_taken_seq2seq).total_seconds()}")
print(f"Speedup:\t{time_taken_seq2seq / time_taken_dynabatch:.2f}x")

Time taken for Dynabatch training:	1099.472182
Time taken for Seq2Seq training:	3623.19785
Speedup:	3.30x
